# Mexico Real Estate Price Analysis
## Part 1: Getting to Know the Data

> **Why does the shape of your data determine everything that follows?**
>
> Before you can visualize, correlate, or model anything, the data must be in the
> right form. Most real-world datasets are *not* in that form — they're built for
> the person entering the data, not for computational analysis. This notebook
> establishes the rules that govern *tidy data* and the pandas tools that enforce
> them.

**Research Question:** Are property prices in Mexico more influenced by property
size or by location?


In [28]:
# Import pandas for data manipulation
import pandas as pd

In [29]:
# Define the path to our data file
path_1 = "../data/mexico-real-estate-1.csv"

In [30]:
# Load the dataset using the variable we defined
(
    pd.read_csv(path_1)
)

,property_type,state,lat,lon,area_m2,price_usd
0,house,Estado de México,19.56,-99.23,150.00,"$67,965.56"
1,house,Nuevo León,25.69,-100.20,186.00,"$63,223.78"
2,apartment,Guerrero,16.77,-99.76,82.00,"$84,298.37"
3,apartment,Guerrero,16.83,-99.91,150.00,"$94,308.80"
4,house,Veracruz de Ignacio de la Llave,NaN,NaN,175.00,"$94,835.67"
...,...,...,...,...,...,...
695,house,Morelos,NaN,NaN,310.00,"$237,089.17"
696,house,Yucatán,21.05,-89.56,334.00,"$137,017.34"
697,house,Yucatán,21.34,-89.26,130.00,"$110,404.35"
698,apartment,Nuevo León,NaN,NaN,155.00,"$184,446.42"


> 📊 **Reading the raw DataFrame output**
>
> You should see a table with columns including `property_type`, `state`, `lat`,
> `lon`, `area_m2`, and `price_usd`. This is the raw data exactly as stored in
> the CSV — no modifications have been made yet.
>
> **What to notice immediately:**
>
> - **`price_usd` shows values like `"$55,000.00"`** — that dollar sign and comma
>   make this column a *text string*, not a number. You cannot compute
>   `df["price_usd"].mean()` until we strip those characters and convert to float.
> - **`lat` and `lon` may show `NaN`** — `NaN` stands for "Not a Number" and is
>   pandas' representation of a missing value. Rows with `NaN` coordinates cannot
>   be used in any geographic analysis.
> - **The index (the unlabeled leftmost column)** starts at 0 and counts up. This
>   is the default integer index that pandas assigns automatically when reading a
>   CSV. It is not a data column.
> - **The data type of `price_usd`** is not yet obvious from looking at the values
>   — it could be stored as text or as a number that happens to include special
>   characters. We need the inspection ritual to find out.
>
> ➡️ Before cleaning anything, we must understand exactly what we are working with.


In [31]:
(
    pd.read_csv(path_1)
    .head()
)

,property_type,state,lat,lon,area_m2,price_usd
0,house,Estado de México,19.56,-99.23,150.00,"$67,965.56"
1,house,Nuevo León,25.69,-100.20,186.00,"$63,223.78"
2,apartment,Guerrero,16.77,-99.76,82.00,"$84,298.37"
3,apartment,Guerrero,16.83,-99.91,150.00,"$94,308.80"
4,house,Veracruz de Ignacio de la Llave,NaN,NaN,175.00,"$94,835.67"


> 📊 **Reading the `.head()` output — Step 1 of the inspection ritual**
>
> `.head()` shows the first 5 rows (by default). This gives you an immediate
> visual picture of the data before you look at any statistics.
>
> - **Column names** — confirm that the six expected variables are present:
>   `property_type`, `state`, `lat`, `lon`, `area_m2`, `price_usd`.
> - **`price_usd` format** — if you see `"$55,000.00"` (with quotes, dollar sign,
>   and comma), this column is stored as text, not a number.
> - **`lat` / `lon`** — some rows may show `NaN` (missing coordinates). Note which
>   rows they are, even if you cannot yet count them.
> - **`property_type`** — values should be strings like `"house"` or `"apartment"`.
> - **The index** — starts at 0, confirming pandas assigned a default integer index.
>
> `.head()` gives you the *shape* of the data visually, but it only shows 5 rows.
> You cannot tell from `.head()` alone how many rows are missing, or what the
> overall type profile looks like. For that, we need the next three steps.


In [34]:
(
    pd.read_csv(path_1)
    .shape
)

(700, 6)

> 📊 **Reading the `.shape` output — Step 2 of the inspection ritual**
>
> You should see a tuple such as `(700, 6)`:
> - **700** — total number of property listings in this file (before cleaning).
> - **6** — total number of variables (columns).
>
> These are your **baseline numbers**. After `.dropna()` removes rows with missing
> coordinates, the first number will decrease. After `.concat()` combines all three
> files, the first number will increase (to roughly the sum of all three files).

In [35]:
(
    pd.read_csv(path_1)
    .info()
)

<class 'pandas.DataFrame'>
RangeIndex: 700 entries, 0 to 699
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   property_type  700 non-null    str    
 1   state          700 non-null    str    
 2   lat            583 non-null    float64
 3   lon            583 non-null    float64
 4   area_m2        700 non-null    float64
 5   price_usd      700 non-null    str    
dtypes: float64(3), str(3)
memory usage: 32.9 KB


> 📊 **Reading the `.info()` output — Step 3 of the inspection ritual**
>
> **2 — Dtype column (most important):**
>
> | Dtype | Meaning | Expected for |
> |---|---|---|
> | `float64` | 64-bit floating-point number | `lat`, `lon`, `area_m2`, `price_usd` (after cleaning) |
> | `int64` | 64-bit integer | Counts, categorical codes |
> | `object` | Python string or mixed types | `property_type`, `state`, `price_usd` (before cleaning) |
>
> **What this reveals about our dataset:**
>
> - `price_usd` is `object` — it contains text strings like `"$55,000.00"`, not
>   numbers. This is the primary type bug we need to fix.
> - `lat` and `lon` are `float64` — but the non-null count will be less than the
>   total row count, telling us that some rows are missing coordinates.
> - `property_type` and `state` are `str` — expected, since they hold text.
>
> **3 — Non-null counts:** A column with `583 non-null` in a 700-row DataFrame has
> exactly 117 missing values. You do not need `.isnull().sum()` to know this —
> though Step 5 will confirm it column by column.
>


In [36]:
(
    pd.read_csv(path_1)
    .dtypes
)

property_type        str
state                str
lat              float64
lon              float64
area_m2          float64
price_usd            str
dtype: object

> 📊 **Reading the `.dtypes` output — Step 4 of the inspection ritual**
>
> **For this dataset:** Confirm that `price_usd` shows `object` here. This single
> fact tells you the entire cleaning plan for that column: strip the `$` and `,`
> characters, then convert to `float64` with `.astype(float)`. Two string
> operations and one type cast — that's it.



In [37]:
(
    pd.read_csv(path_1)
    .isnull()
    .sum()
)   

property_type      0
state              0
lat              117
lon              117
area_m2            0
price_usd          0
dtype: int64

> 📊 **Reading the `.isnull().sum()` output — Step 5 of the inspection ritual**

> **Cleaning decision:** We will drop the 117 rows missing coordinates using
> `.dropna()`. This is reasonable because:
> (a) 117/700 ≈ 16.7% — a manageable loss,
> (b) we have no way to impute coordinates meaningfully,
> (c) any geographic analysis would fail anyway on those rows.

## 4. Building the Cleaning Pipeline (Method Chaining)
### 4.2 Step 1: Dropping Missing Values with `.dropna()`

From the inspection ritual, we know 117 rows are missing `lat` / `lon`. We remove
them with `.dropna()`.

In [38]:
# Step 1: Drop missing values
(
    pd.read_csv(path_1)
    .dropna()  # Remove rows where values are missing
    
)

,property_type,state,lat,lon,area_m2,price_usd
0,house,Estado de México,19.56,-99.23,150.00,"$67,965.56"
1,house,Nuevo León,25.69,-100.20,186.00,"$63,223.78"
2,apartment,Guerrero,16.77,-99.76,82.00,"$84,298.37"
3,apartment,Guerrero,16.83,-99.91,150.00,"$94,308.80"
5,house,Yucatán,21.05,-89.54,205.00,"$105,191.37"
...,...,...,...,...,...,...
693,house,Puebla,19.05,-98.28,198.00,"$115,910.26"
694,apartment,Distrito Federal,19.31,-99.17,70.00,"$77,572.89"
696,house,Yucatán,21.05,-89.56,334.00,"$137,017.34"
697,house,Yucatán,21.34,-89.26,130.00,"$110,404.35"


### 4.3 Step 2: Cleaning `price_usd` with `.assign()` and `lambda`

In [39]:
# Step 2: The Complete Chain
(
    pd.read_csv(path_1)
    # 1. Drop rows with missing critical data
    .dropna()
    
    # 2. Create clean numeric columns using `.assign()`
    # We use 'lambda x' to refer to the data at this specific step in the chain
    # We overwrite 'price_usd' by accessing the current df (x in our lambda function) and cleaning the string
    .assign(
        price_usd = lambda x: x["price_usd"]
                          .str.replace("$", "", regex=False)
                          .str.replace(",", "", regex=False)
                          .astype(float)
    )
)

,property_type,state,lat,lon,area_m2,price_usd
0,house,Estado de México,19.56,-99.23,150.00,"67,965.56"
1,house,Nuevo León,25.69,-100.20,186.00,"63,223.78"
2,apartment,Guerrero,16.77,-99.76,82.00,"84,298.37"
3,apartment,Guerrero,16.83,-99.91,150.00,"94,308.80"
5,house,Yucatán,21.05,-89.54,205.00,"105,191.37"
...,...,...,...,...,...,...
693,house,Puebla,19.05,-98.28,198.00,"115,910.26"
694,apartment,Distrito Federal,19.31,-99.17,70.00,"77,572.89"
696,house,Yucatán,21.05,-89.56,334.00,"137,017.34"
697,house,Yucatán,21.34,-89.26,130.00,"110,404.35"


In [40]:
# Step 2: The Complete Chain
df1 = (
    pd.read_csv(path_1)
    # 1. Drop rows with missing critical data
    .dropna()
    
    # 2. Create clean numeric columns using `.assign()`
    # We use 'lambda x' to refer to the data at this specific step in the chain
    # We overwrite 'price_usd' by accessing the current df (x in our lambda function) and cleaning the string
    .assign(
        price_usd = lambda x: x["price_usd"]
                          .str.replace("$", "", regex=False)
                          .str.replace(",", "", regex=False)
                          .astype(float)
    )
)

> `df1` is now your clean, typed, complete first dataset. It is ready to be
> stacked with the other two. We will do that in §5.


## 5. Clean the Rest of the Dataset and Combine

We now have one clean DataFrame (`df1`) from the first CSV file. There are two more
CSV files in the data folder — `mexico-real-estate-2.csv` and
`mexico-real-estate-3.csv` — each covering a different set of property listings.


In [41]:
# Define the path to our data file
path_2 = "../data/mexico-real-estate-2.csv"

In [42]:
# Define the path to our data file
path_3 = "../data/mexico-real-estate-3.csv"

### What is different about `df2`?

`df2` (`mexico-real-estate-2.csv`) stores prices in **Mexican pesos** (`price_mxn`)
instead of US dollars. We must:
1. Convert `price_mxn` to `price_usd` by dividing by the exchange rate (19 MXN/USD).`


In [43]:
# cleaning process using method chaining
df2 = (
    pd.read_csv(path_2)

     # Create "price_usd" column (19 pesos to the dollar in 2014)
    .assign(price_usd = lambda x: x["price_mxn"].div(19))

    # Drop "price_mxn" column
    .drop(columns=["price_mxn"])

    # Drop null values
    .dropna()
)

# Print object type, shape, and head
print(f"df2 type: {type(df2)}")
print(f"df2 shape: {df2.shape}")
df2.head()

df2 type: <class 'pandas.DataFrame'>
df2 shape: (571, 6)


,property_type,state,lat,lon,area_m2,price_usd
0,apartment,Nuevo León,25.72,-100.35,72.00,"68,421.05"
2,house,Morelos,23.63,-102.55,360.00,"278,947.37"
6,apartment,Estado de México,19.27,-99.57,85.00,"65,789.47"
7,house,San Luis Potosí,22.14,-101.00,158.00,"111,578.95"
8,apartment,Distrito Federal,19.39,-99.13,65.00,"39,904.74"


### What is different about `df3`?

The fixing process is two steps per column:
1. **`.str.split(separator, expand=True)`** — splits the string and expands the
   result into separate columns (indexed 0, 1, 2, ...).
2. **`[index]`** — selects the specific piece we need, then `.astype(float)` if
   the result should be a number.


In [44]:
# cleaning process using method chaining
df3 = (
    pd.read_csv(path_3)
    # Drop null values from df3
    .dropna()
    # Create "lat" and "lon" columns for df3
    .assign(lat=lambda x: x["lat-lon"]
                .str.split(",", expand=True)[0]           
                .astype(float),
            lon=lambda x: x["lat-lon"]
                .str.split(",", expand=True)[1]           
                .astype(float),
            # Create "state" column for df3
            state=lambda x: x["place_with_parent_names"]
            .str.split("|", expand=True)[2]
           )
    
    # Drop "place_with_parent_names" and "lat-lon" from df3
    .drop(columns=["place_with_parent_names", "lat-lon"])
            
    
)

# Print object type, shape, and head
print("df3 type:", type(df3))
print("df3 shape:", df3.shape)
df3.head()

df3 type: <class 'pandas.DataFrame'>
df3 shape: (582, 6)


,property_type,area_m2,price_usd,lat,lon,state
0,apartment,71.00,"48,550.59",19.53,-99.15,Distrito Federal
1,house,233.00,"168,636.73",19.26,-99.57,Estado de México
2,house,300.00,"86,932.69",19.27,-99.67,Estado de México
4,apartment,84.00,"68,508.67",19.51,-96.87,Veracruz de Ignacio de la Llave
5,house,175.00,"102,763.00",20.69,-103.37,Jalisco


In [45]:
# Check if all column sets are identical
columns_match = set(df1.columns) == set(df2.columns) == set(df3.columns)

if columns_match:
    print("All DataFrames have the same columns.")
else:
    print("Column mismatch detected!")

All DataFrames have the same columns.


In [46]:
# Stack df1, df2, and df3 vertically
df = pd.concat([df1, df2, df3], ignore_index=True)

# Verify the result
print(f"Combined Shape: {df.shape}")
df.head()

Combined Shape: (1736, 6)


,property_type,state,lat,lon,area_m2,price_usd
0,house,Estado de México,19.56,-99.23,150.00,"67,965.56"
1,house,Nuevo León,25.69,-100.20,186.00,"63,223.78"
2,apartment,Guerrero,16.77,-99.76,82.00,"84,298.37"
3,apartment,Guerrero,16.83,-99.91,150.00,"94,308.80"
4,house,Yucatán,21.05,-89.54,205.00,"105,191.37"


In [47]:
# Save your cleaned combined dataset
df.to_csv("../data/mexico-real-estate-combined-clean.csv", index=False)

## 6. Descriptive Statistics

> **`.describe()` — your first statistical snapshot of the combined dataset**
>
> With the three files merged and saved, we can now ask: *what does the overall
> distribution of prices and areas look like?* `.describe()` answers this question
> with eight summary statistics per numeric column, computed on the combined
> 1,700+ row dataset.

In [48]:
# Summarize the numeric columns
summary = df[["price_usd", "area_m2"]].describe()

# # Format for readability
pd.options.display.float_format = '{:,.2f}'.format

summary

,price_usd,area_m2
count,"1,736.00","1,736.00"
mean,"115,331.98",170.26
std,"65,426.17",80.59
min,"33,157.89",60.00
25%,"65,789.47",101.75
50%,"99,262.13",156.00
75%,"150,846.66",220.00
max,"326,733.66",385.00
